In [ ]:
# Spinner utility for long-running steps (no extra deps)
from contextlib import contextmanager
from IPython.display import display, HTML
import threading, time


def _spinner_loop(text: str, stop_event: threading.Event):
    frames = ['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏']
    box = display(HTML(f"<span>{text} <code>…</code></span>"), display_id=True)
    i = 0
    start = time.perf_counter()
    while not stop_event.is_set():
        elapsed = time.perf_counter() - start
        box.update(HTML(f"<span>{text} <code>{frames[i % len(frames)]}</code> <small>({elapsed:0.1f}s)</small></span>"))
        i += 1
        time.sleep(0.1)
    elapsed = time.perf_counter() - start
    box.update(HTML(f"<span>{text} <strong>✓ Done</strong> <small>({elapsed:0.1f}s)</small></span>"))


@contextmanager
def spinner(text: str = "Working..."):
    stop = threading.Event()
    t = threading.Thread(target=_spinner_loop, args=(text, stop), daemon=True)
    t.start()
    try:
        yield
    finally:
        stop.set()
        t.join()

## Progress indicator
A lightweight spinner will show while the experiment is running.

In [ ]:
# Set working directory to repo root and list experiments in myexperiments/
from pathlib import Path
import os, datetime

# Detect repo root by walking up to find pyproject.toml or .git
cwd = Path.cwd()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "pyproject.toml").exists() or (p / ".git").exists():
        repo_root = p
        break
repo_root = repo_root or cwd
os.chdir(repo_root)
print(f"Working directory set to repo root: {repo_root}")

# List experiments
exp_root = repo_root / "myexperiments"
rows = []
if exp_root.exists():
    for run_dir in sorted(exp_root.iterdir()):
        if run_dir.is_dir():
            summary = next(run_dir.rglob("reports/summary/summary.json"), None)
            mtime = datetime.datetime.fromtimestamp(run_dir.stat().st_mtime)
            rows.append({
                "name": run_dir.name,
                "path": str(run_dir),
                "summary_path": str(summary) if summary else "",
                "modified": mtime.isoformat(timespec="seconds"),
            })
    if not rows:
        print("No experiments found yet under 'myexperiments'.")
else:
    print("No 'myexperiments' directory found at repo root yet.")

try:
    import pandas as pd
    if rows:
        df = pd.DataFrame(rows).sort_values("modified", ascending=False)
        display(df)
except Exception:
    for r in rows:
        print(f"- {r['name']}  (summary: {r['summary_path'] or 'none'})")

## Project root & experiments
This notebook will first switch the working directory to the repository root and list any existing experiments under `myexperiments/`.

## Configure API credentials
Provide your API key, channel, and region. This cell sets the session environment and can also persist a config file at `~/.config/mips/atlaspy/config.json` for reuse.

In [ ]:
# Configure API credentials for ATLAS Explorer
# Auto-detect from env/.env or user config first; if none, fill values below.

from pathlib import Path
import os, json
from dotenv import load_dotenv

load_dotenv()
CONFIG_ENV = "MIPS_ATLAS_CONFIG"
cfg_file = Path.home() / ".config/mips/atlaspy/config.json"

config_detected = False
if os.environ.get(CONFIG_ENV):
    print("Using credentials from MIPS_ATLAS_CONFIG (env/.env).")
    config_detected = True
elif cfg_file.exists():
    try:
        data = json.loads(cfg_file.read_text())
        os.environ[CONFIG_ENV] = f"{data['apikey']}:{data['channel']}:{data['region']}"
        print(f"Using credentials from {cfg_file}")
        config_detected = True
    except Exception as e:
        print(f"Found {cfg_file} but couldn't parse: {e}")

if not config_detected:
    ae_apikey = ""
    ae_channel = ""
    ae_region = ""
    persist_to_file = True

    if not ae_apikey or not ae_channel or not ae_region:
        print("No credentials detected. Set ae_apikey/ae_channel/ae_region above and re-run.")
    else:
        os.environ[CONFIG_ENV] = f"{ae_apikey}:{ae_channel}:{ae_region}"
        print("Session env set: MIPS_ATLAS_CONFIG")
        if persist_to_file:
            cfg_file.parent.mkdir(parents=True, exist_ok=True)
            cfg_file.write_text(json.dumps({"apikey": ae_apikey, "channel": ae_channel, "region": ae_region}, indent=2))
            print(f"Wrote config file: {cfg_file}")
        else:
            print("Skipping config file write (persist_to_file=False)")

# ATLAS Explorer: Multicore + Baseline Diff Report (Notebook)

Run a multicore experiment, then build a ZIP bundle with JSON, Markdown, and rich HTML reports. Optionally include a baseline diff.

## Prerequisites
- Configure credentials once: run the CLI configurator or set `MIPS_ATLAS_CONFIG`.
- Example CLI: `uv run atlasexplorer/atlasexplorer.py configure`.
- Ensure example ELFs exist (defaults use files in `resources/`).

In [ ]:
# Settings — adjust as needed
elfs = ["resources/mandelbrot_rv64_O0.elf", "resources/memcpy_rv64.elf"]
expdir = "myexperiments"
core = "I8500_(2_threads)"
channel = "development"
apikey = None   # or a string
region = None   # or a string
verbose = False

# Optional baseline summary.json (set to a path string to enable diffs)
baseline = None  # e.g., 'atlas-explorer-experiments/.../reports/summary/summary.json'
# Output ZIP bundle path
bundle_out = "diff_bundle.zip"

In [ ]:
# Environment setup
import os, locale, tempfile, zipfile
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
locale.setlocale(locale.LC_ALL, "")

config_env = os.environ.get("MIPS_ATLAS_CONFIG")
home_dir = os.path.expanduser("~")
config_file = os.path.join(home_dir, ".config", "mips", "atlaspy", "config.json")
if not apikey and not config_env and not os.path.exists(config_file):
    print("Tip: run 'uv run atlasexplorer/atlasexplorer.py configure' or set MIPS_ATLAS_CONFIG.")

In [ ]:
# Run the experiment
from atlasexplorer.atlasexplorer import AtlasExplorer, Experiment

aeinst = AtlasExplorer(apikey, channel, region, verbose=verbose)
experiment = Experiment(expdir, aeinst, verbose=verbose)
for elf in elfs:
    experiment.addWorkload(os.path.abspath(elf))
experiment.setCore(core)

# Show spinner during the long-running execution
with spinner("Running experiment (this can take several minutes)..."):
    experiment.run()

# Find newest summary.json
exp_path = Path(expdir)
summary_candidates = list(exp_path.rglob("summary/summary.json"))
if not summary_candidates:
    raise FileNotFoundError("No summary.json found under experiment directory")
summary_candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
summary_path = summary_candidates[0]
print(f"Produced summary: {summary_path}")

In [ ]:
# Build report bundle (JSON, MD, rich HTML), with optional baseline diff
from atlasexplorer.reporting.parser import parse_summary_json
from atlasexplorer.reporting.derive import apply_derivations
from atlasexplorer.reporting.thresholds import apply_thresholds
from atlasexplorer.reporting.export import export_json, export_markdown, export_rich_html

main_report = apply_thresholds(apply_derivations(parse_summary_json(str(summary_path))))
base_report = None
if baseline:
    base_report = apply_derivations(parse_summary_json(str(Path(baseline).resolve())))

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)
    json_path = tmpdir_path / 'report.json'
    md_path = tmpdir_path / 'report.md'
    rich_path = tmpdir_path / 'report_rich.html'
    export_json(main_report, json_path)
    export_markdown(main_report, md_path)
    export_rich_html(main_report, rich_path)

    if base_report:
        lines = ["# Baseline Comparison\n", "| Metric | New | Baseline | Δ | Δ% |", "|--------|----|----------|---|----|"]
        base_map = {m.name: m for m in base_report.metrics}
        for m in main_report.metrics:
            if m.name in base_map and m.value is not None and base_map[m.name].value is not None:
                new_v = m.value
                old_v = base_map[m.name].value
                delta = new_v - old_v
                pct = (delta / old_v * 100) if old_v not in (0, None) else float('inf')
                lines.append(f"| {m.name} | {new_v:.4g} | {old_v:.4g} | {delta:+.4g} | {pct:+.2f}% |")
        (tmpdir_path / 'baseline_diff.md').write_text("\n".join(lines))

    bundle_out_path = Path(bundle_out).resolve()
    bundle_out_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_out_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for f in tmpdir_path.iterdir():
            zf.write(f, arcname=f.name)

print(f"Bundle written to: {bundle_out_path}")

In [ ]:
# Quick ad-hoc analysis as a DataFrame
from atlasexplorer.reporting.parser import parse_summary_json
from atlasexplorer.reporting.derive import apply_derivations
import pandas as pd

report = apply_derivations(parse_summary_json(str(summary_path)))
df = pd.DataFrame([m.model_dump() for m in report.metrics])
df.head()